In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# =====================================
# 1. 路径
# =====================================

input_file = r"H:\图像标记\12个波段结果\12个特征波段_波长版数据.xlsx"

output_dir = r"H:\图像标记\12个波段结果\2"

os.makedirs(output_dir, exist_ok=True)

# =====================================
# 2. 读取数据
# =====================================

print("读取数据...")

df = pd.read_excel(input_file)

feature_cols = df.columns[1:]

print("数据维度:", df.shape)

# =====================================
# 3. 类别名称
# =====================================

label_map = {
    0: 'Rice',
    1: 'Barnyardgrass',
    2: 'Leptochloa'
}

# =====================================
# 4. 显著性星号函数
# =====================================

def significance_star(p):

    if p < 0.0001:
        return '****'
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

# =====================================
# 5. 方差分析 + Tukey检验
# =====================================

print("开始方差分析...")

summary_results = []

tukey_all_results = []

for feature in feature_cols:

    # ==========================
    # 三类数据
    # ==========================

    rice = df[df['Label']==0][feature]
    barn  = df[df['Label']==1][feature]
    lept  = df[df['Label']==2][feature]

    # ==========================
    # ANOVA
    # ==========================

    F, p = f_oneway(rice, barn, lept)

    overall_sig = significance_star(p)

    # ==========================
    # Tukey两两比较
    # ==========================

    tukey = pairwise_tukeyhsd(
        endog=df[feature],
        groups=df['Label'],
        alpha=0.05
    )

    tukey_df = pd.DataFrame(
        data=tukey.summary().data[1:],
        columns=tukey.summary().data[0]
    )

    tukey_df.insert(0, 'Wavelength', feature)

    tukey_all_results.append(tukey_df)

    # ==========================
    # 提取三组比较结果
    # ==========================

    # 0 vs 1
    p01 = tukey_df.iloc[0]['p-adj']
    sig01 = significance_star(p01)

    # 0 vs 2
    p02 = tukey_df.iloc[1]['p-adj']
    sig02 = significance_star(p02)

    # 1 vs 2
    p12 = tukey_df.iloc[2]['p-adj']
    sig12 = significance_star(p12)

    # ==========================
    # 判断区分能力
    # ==========================

    sig_count = sum([
        sig01 != 'ns',
        sig02 != 'ns',
        sig12 != 'ns'
    ])

    if sig_count == 3:
        conclusion = '可显著区分三类样本'

    elif sig_count == 2:
        conclusion = '可区分某两类与另一类'

    elif sig_count == 1:
        conclusion = '仅部分区分'

    else:
        conclusion = '区分能力较弱'

    # ==========================
    # 保存汇总结果
    # ==========================

    summary_results.append({

        'Wavelength': feature,

        'Rice_Mean±Std':
            f'{rice.mean():.4f} ± {rice.std():.4f}',

        'Barnyardgrass_Mean±Std':
            f'{barn.mean():.4f} ± {barn.std():.4f}',

        'Leptochloa_Mean±Std':
            f'{lept.mean():.4f} ± {lept.std():.4f}',

        'F_value':
            round(F,4),

        'P_value':
            p,

        'Overall_Significance':
            overall_sig,

        'Rice_vs_Barnyardgrass':
            sig01,

        'Rice_vs_Leptochloa':
            sig02,

        'Barnyardgrass_vs_Leptochloa':
            sig12,

        'Conclusion':
            conclusion
    })

# =====================================
# 6. 保存总表
# =====================================

summary_df = pd.DataFrame(summary_results)

# 按F值排序
summary_df = summary_df.sort_values(
    by='F_value',
    ascending=False
)

summary_path = os.path.join(
    output_dir,
    '12个波长_显著性分析总表.xlsx'
)

summary_df.to_excel(summary_path, index=False)

print("总表已保存:")
print(summary_path)

# =====================================
# 7. 保存Tukey详细结果
# =====================================

tukey_final_df = pd.concat(
    tukey_all_results,
    ignore_index=True
)

tukey_path = os.path.join(
    output_dir,
    'Tukey两两比较详细结果.xlsx'
)

tukey_final_df.to_excel(
    tukey_path,
    index=False
)

print("Tukey结果已保存")

# =====================================
# 8. 生成结果说明
# =====================================

txt_path = os.path.join(
    output_dir,
    '结果说明.txt'
)

with open(txt_path, 'w', encoding='utf-8') as f:

    f.write('显著性等级说明:\\n\\n')

    f.write('*     P < 0.05\\n')
    f.write('**    P < 0.01\\n')
    f.write('***   P < 0.001\\n')
    f.write('****  P < 0.0001\\n')
    f.write('ns    not significant\\n\\n')

    f.write('结果解读:\\n\\n')

    f.write('1. Overall_Significance\\n')
    f.write('   表示三类整体是否存在显著差异\\n\\n')

    f.write('2. Rice_vs_Barnyardgrass\\n')
    f.write('   表示水稻与稗草之间是否显著不同\\n\\n')

    f.write('3. Rice_vs_Leptochloa\\n')
    f.write('   表示水稻与千金子之间是否显著不同\\n\\n')

    f.write('4. Barnyardgrass_vs_Leptochloa\\n')
    f.write('   表示稗草与千金子之间是否显著不同\\n\\n')

    f.write('5. Conclusion\\n')
    f.write('   表示该波长的类别区分能力\\n')

print("\\n全部完成！")

# =====================================
# 9. 输出最强区分波长
# =====================================

print("\\n========== 最强区分波长 ==========")

top5 = summary_df.head(5)

print(top5[
    [
        'Wavelength',
        'F_value',
        'Conclusion'
    ]
])


In [ ]:
# 严格对应三组两两比较的箱线图+显著性标注
# 优化：所有边框/连线统一 #000000 黑色 + 线宽 1 + 四周边框全开
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ===================== 路径配置（和你原来的完全一致） =====================
TRAIN_PATH = r"H:\图像标记\train_test\sg三类精准平衡后_训练集.xlsx"
BAND_TABLE_PATH = r"H:\图像标记\12个波段结果\12个特征波段_波长对照表.xlsx"
SIG_TABLE_PATH = r"H:\图像标记\12个波段结果\2\12个波长_显著性分析总表.xlsx"
SAVE_DIR = r"H:\图像标记\12个波段结果"
os.makedirs(SAVE_DIR, exist_ok=True)

# 类别配置（严格对应：水稻、稗草、千金子）
label_map = {0: "水稻", 1: "稗草", 2: "千金子"}
colors = ["#D73F64", "#FC9D48", "#0083A4"]  # 分别对应水稻、稗草、千金子
feature_bands = [
    'Band_5','Band_31','Band_35','Band_40','Band_44','Band_49',
    'Band_73','Band_76','Band_91','Band_93','Band_94','Band_95'
]

# ===================== 读取波段对照表（波长和箱线一一对应） =====================
band_df = pd.read_excel(BAND_TABLE_PATH)
band_wave_dict = {}
for _, row in band_df.iterrows():
    band_name = str(row.iloc[0]).strip()
    wave = row.iloc[1]
    band_wave_dict[band_name] = f"{wave}nm"  

wave_labels = [band_wave_dict[b] for b in feature_bands]

# ===================== 读取原始数据（生成箱线图用） =====================
df = pd.read_excel(TRAIN_PATH)
df_selected = df[['Label'] + feature_bands].copy()
df_selected['Label'] = df_selected['Label'].map(label_map)
df_melt = pd.melt(df_selected, id_vars="Label", var_name="特征波段", value_name="Reflectance")

# ===================== 读取显著性表（严格对应三组比较的星号） =====================
sig_df = pd.read_excel(SIG_TABLE_PATH)
wave_col = "Wavelength"  # 你表中第一列的列名

# 自动识别三列显著性结果（不用手动改列名）
rb_col, rl_col, bl_col = None, None, None
for c in sig_df.columns:
    cs = str(c).lower()
    if "rice" in cs and "barnyard" in cs:
        rb_col = c  # 水稻 vs 稗草 列
    elif "rice" in cs and "lepto" in cs:
        rl_col = c  # 水稻 vs 千金子 列
    elif "barnyard" in cs and "lepto" in cs:
        bl_col = c  # 稗草 vs 千金子 列

# 构建波长→显著性结果的字典（严格对应三组）
sig_dict = {}
for _, row in sig_df.iterrows():
    w = str(row[wave_col]).strip()
    sig_dict[w] = {
        "水稻vs稗草": str(row[rb_col]).strip() if pd.notna(row[rb_col]) else "",
        "水稻vs千金子": str(row[rl_col]).strip() if pd.notna(row[rl_col]) else "",
        "稗草vs千金子": str(row[bl_col]).strip() if pd.notna(row[bl_col]) else ""
    }

# ===================== 绘制箱线图 =====================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 加宽画布，给显著性连线留出足够空间
plt.figure(figsize=(16, 7), dpi=300)
TICK_SIZE = 16

ax = sns.boxplot(
    data=df_melt,
    x="特征波段",
    y="Reflectance",
    hue="Label",
    palette=colors,
    linewidth=1,
    saturation=1,
    fliersize=0
)

# 基础样式调整（和你原来的风格一致）
plt.xlabel("")
plt.ylabel("")
plt.legend().remove()
plt.grid(False)

# X轴标签旋转，避免长文字重叠
plt.xticks(
    ticks=range(len(feature_bands)),
    labels=wave_labels,
    fontsize=TICK_SIZE,
    rotation=45,
    ha='right'
)
plt.yticks(fontsize=TICK_SIZE)

# ===================== 【已加入】四周边框：#000000 黑色，线宽 1 =====================
ax.spines['top'].set_visible(True)    # 显示顶部边框
ax.spines['right'].set_visible(True)  # 显示右侧边框
ax.spines['left'].set_visible(True)   # 显示左侧边框
ax.spines['bottom'].set_visible(True) # 显示底部边框

# 所有边框统一颜色和宽度
ax.spines['top'].set_color('#000000')
ax.spines['right'].set_color('#000000')
ax.spines['left'].set_color('#000000')
ax.spines['bottom'].set_color('#000000')

ax.spines['top'].set_linewidth(1.5)
ax.spines['right'].set_linewidth(1.5)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# ---------------------- 核心：严格对应三组比较的连线+星号 ----------------------
y_max = df_melt['Reflectance'].max()
box_offsets = [-0.22, 0, 0.22]
line_step = 0.07 * y_max

# 统一颜色 + 线宽
LINE_COLOR = '#000000'
LINE_WIDTH = 1.5

for band_idx, wave in enumerate(wave_labels):
    if wave not in sig_dict:
        continue
    sig = sig_dict[wave]
    x_base = band_idx
    
    x_水稻 = x_base + box_offsets[0]
    x_稗草 = x_base + box_offsets[1]
    x_千金子 = x_base + box_offsets[2]
    
    current_y = y_max + 0.05 * y_max

    # 1. 水稻 vs 稗草
    if sig["水稻vs稗草"]:
        ax.plot([x_水稻, x_稗草], [current_y, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.plot([x_水稻, x_水稻], [current_y - 0.015*y_max, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.plot([x_稗草, x_稗草], [current_y - 0.015*y_max, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.text((x_水稻 + x_稗草)/2, current_y + 0.01*y_max, sig["水稻vs稗草"],
                ha='center', va='bottom', fontsize=16, fontweight='bold', color=LINE_COLOR)
        current_y += line_step

    # 2. 水稻 vs 千金子
    if sig["水稻vs千金子"]:
        ax.plot([x_水稻, x_千金子], [current_y, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.plot([x_水稻, x_水稻], [current_y - 0.015*y_max, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.plot([x_千金子, x_千金子], [current_y - 0.015*y_max, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.text((x_水稻 + x_千金子)/2, current_y + 0.01*y_max, sig["水稻vs千金子"],
                ha='center', va='bottom', fontsize=16, fontweight='bold', color=LINE_COLOR)
        current_y += line_step

    # 3. 稗草 vs 千金子
    if sig["稗草vs千金子"]:
        ax.plot([x_稗草, x_千金子], [current_y, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.plot([x_稗草, x_稗草], [current_y - 0.015*y_max, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.plot([x_千金子, x_千金子], [current_y - 0.015*y_max, current_y], color=LINE_COLOR, linewidth=LINE_WIDTH)
        ax.text((x_稗草 + x_千金子)/2, current_y + 0.01*y_max, sig["稗草vs千金子"],
                ha='center', va='bottom', fontsize=16, fontweight='bold', color=LINE_COLOR)
        current_y += line_step

plt.subplots_adjust(top=0.8)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "箱线图_三组两两显著性标注字号16.png"), dpi=300, bbox_inches='tight')
plt.close()

print("✅ 三组两两带连线+星号+四周边框的箱线图已生成！")